# 05 - Model Comparison

This notebook compiles SOH and RUL benchmarks, saves `results/metrics.csv`, and creates final comparison visualizations.

In [1]:
from pathlib import Path
import sys
import random
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
METRICS_PATH = PROJECT_ROOT / 'results' / 'metrics.csv'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## Load feature data and run both benchmark suites

In [2]:
import numpy as np
from src.models import run_soh_benchmark, run_rul_benchmark
from src.evaluation import save_metrics

feature_df = pd.read_csv(PROCESSED_DIR / 'features_all.csv')

soh_metrics, soh_results = run_soh_benchmark(feature_df=feature_df, models_dir=MODELS_DIR)
rul_metrics, rul_results = run_rul_benchmark(feature_df=feature_df, models_dir=MODELS_DIR, include_transformer=True)

metrics_df = save_metrics(soh_metrics + rul_metrics, METRICS_PATH)
metrics_df

,Model,Task,RMSE,MAE,MAPE,R2,Train_Time_s
0,Random Forest Regressor,RUL,6.373208,5.091485,17.973037,0.962434,0.269855
1,XGBoost Regressor,RUL,6.385948,4.935395,15.623504,0.962284,0.849606
2,Transformer Encoder,RUL,6.583761,5.866489,50.553942,0.951852,5.529124
3,LSTM Neural Network,RUL,33.463212,29.424875,198.333743,-0.243854,1.220792
4,Linear Regression,SOH,0.116027,0.099809,0.127294,0.999769,0.000396
5,XGBoost Regressor,SOH,0.565523,0.367674,0.472972,0.994523,4.140315
6,Random Forest Regressor,SOH,0.714788,0.408414,0.530208,0.991250,0.175467
7,Ridge Regression,SOH,0.943786,0.885303,1.143149,0.984746,0.000322
8,CNN-BiLSTM,SOH,6.894650,6.183861,8.181457,0.013460,0.888562
9,LSTM Neural Network,SOH,6.941668,6.239934,8.040798,-0.000041,1.127488


## RMSE comparison chart

In [3]:
from src.visualisation import plot_rmse_comparison

plot_rmse_comparison(metrics_df, FIG_DIR / '08_rmse_comparison_soh_rul.png')
print('Saved RMSE comparison chart.')

Saved RMSE comparison chart.


## Best-model scatter: predicted vs actual

In [4]:
from src.visualisation import plot_predicted_vs_actual_scatter

best_row = metrics_df.loc[metrics_df['RMSE'].idxmin()]
best_model = str(best_row['Model'])
best_task = str(best_row['Task'])
source = soh_results if best_task == 'SOH' else rul_results
best_payload = source[best_model]

plot_predicted_vs_actual_scatter(
    y_true=np.asarray(best_payload['y_true']),
    y_pred=np.asarray(best_payload['predictions']),
    title=f'Predicted vs Actual ({best_task} - {best_model})',
    output_path=FIG_DIR / '09_best_model_predicted_vs_actual_scatter.png',
)
print('Saved best-model scatter plot.')

Saved best-model scatter plot.


## Feature importance of best tree model

In [5]:
from src.visualisation import plot_feature_importance

candidate_payloads = []
for payload in list(soh_results.values()) + list(rul_results.values()):
    if hasattr(payload.get('model'), 'feature_importances_'):
        candidate_payloads.append(payload)

if candidate_payloads:
    best_tree_payload = min(candidate_payloads, key=lambda x: x['metrics']['RMSE'])
    model = best_tree_payload['model']
    importance_df = pd.DataFrame(
        {
            'feature': best_tree_payload['feature_columns'],
            'importance': model.feature_importances_,
        }
    )
    plot_feature_importance(
        importance_df=importance_df,
        output_path=FIG_DIR / '10_best_tree_feature_importance.png',
        top_n=15,
    )
    display(importance_df.sort_values('importance', ascending=False).head(10))
else:
    print('No tree-based model found for feature importance.')

,feature,importance
17,capacity_lag_1,0.501629
13,energy_discharged,0.372927
6,energy_charged_Wh,0.071389
18,capacity_lag_3,0.036621
19,capacity_lag_5,0.008660
2,avg_current,0.004095
5,discharge_time,0.002222
12,discharge_duration,0.001507
16,cycle_efficiency,0.000333
11,temperature_rise,0.000103


## Learning curves for all neural network models

In [6]:
from src.visualisation import plot_learning_curve

for model_name, payload in soh_results.items():
    if 'history' in payload:
        safe_name = model_name.lower().replace(' ', '_').replace('-', '_')
        plot_learning_curve(
            history=payload['history'],
            title=f'SOH Learning Curve - {model_name}',
            output_path=FIG_DIR / f'learning_curve_soh_{safe_name}.png',
        )

for model_name, payload in rul_results.items():
    if 'history' in payload:
        safe_name = model_name.lower().replace(' ', '_').replace('-', '_')
        plot_learning_curve(
            history=payload['history'],
            title=f'RUL Learning Curve - {model_name}',
            output_path=FIG_DIR / f'learning_curve_rul_{safe_name}.png',
        )

print('Saved neural network learning curves.')

Saved neural network learning curves.
